In [1]:
# Import libararies
import pandas as pd

from google.colab import files

In [2]:
# Read data
df = pd.read_csv("performance_data_pipeline_cleaned.csv")

In [3]:
# Display data
df

,CaseID,OfficerID,OfficerName,Team,DateCreated,DateClosed,Status,ResolutionHours,SLA_Target_Hours,SLA_Met,CSAT,ResolutionDays,Month,Year,Day,Week,CaseAge,SLA_Met_Flag,CSAT_Category,Resolution_Category
0,1001,O004,Officer 4,Team B,2024-04-26,2024-04-27,Closed,36,48,Yes,3.9,1.0,April,2024,Friday,17,1.0,1.0,Low,24-48 hrs
1,1002,O005,Officer 5,Team C,2024-04-24,NaN,Open,73,48,Pending,0.0,NaN,April,2024,Wednesday,17,811.0,NaN,NaN,>72 hrs
2,1003,O005,Officer 5,Team C,2024-04-12,2024-04-14,Closed,60,48,No,4.7,2.0,April,2024,Friday,15,2.0,0.0,High,49-72 hrs
3,1004,O003,Officer 3,Team B,2024-04-29,2024-05-02,Closed,74,48,No,4.3,3.0,April,2024,Monday,18,3.0,0.0,Medium,>72 hrs
4,1005,O003,Officer 3,Team B,2024-04-03,2024-04-03,Closed,16,48,Yes,4.3,0.0,April,2024,Wednesday,14,0.0,1.0,Medium,<24 hrs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1196,O004,Officer 4,Team B,2024-04-20,NaN,Open,34,48,Pending,0.0,NaN,April,2024,Saturday,16,815.0,NaN,NaN,24-48 hrs
196,1197,O001,Officer 1,Team A,2024-04-22,2024-04-24,Closed,71,48,No,4.6,2.0,April,2024,Monday,17,2.0,0.0,High,49-72 hrs
197,1198,O003,Officer 3,Team B,2024-04-25,2024-04-25,Closed,10,48,Yes,3.5,0.0,April,2024,Thursday,17,0.0,1.0,Low,<24 hrs
198,1199,O002,Officer 2,Team A,2024-04-15,2024-04-18,Closed,72,48,No,4.1,3.0,April,2024,Monday,16,3.0,0.0,Medium,49-72 hrs


In [4]:
# Check data size
df.shape

(200, 20)

In [5]:
# Check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CaseID               200 non-null    int64  
 1   OfficerID            200 non-null    object 
 2   OfficerName          200 non-null    object 
 3   Team                 200 non-null    object 
 4   DateCreated          200 non-null    object 
 5   DateClosed           183 non-null    object 
 6   Status               200 non-null    object 
 7   ResolutionHours      200 non-null    int64  
 8   SLA_Target_Hours     200 non-null    int64  
 9   SLA_Met              200 non-null    object 
 10  CSAT                 200 non-null    float64
 11  ResolutionDays       183 non-null    float64
 12  Month                200 non-null    object 
 13  Year                 200 non-null    int64  
 14  Day                  200 non-null    object 
 15  Week                 200 non-null    int

In [6]:
# Create fact table
fact_cases = df[
[
"CaseID",
"OfficerID",
"DateCreated",
"Status",
"ResolutionHours",
"SLA_Target_Hours",
"SLA_Met_Flag",
"CSAT",
"CaseAge"
]
]

In [7]:
# Check fact table
fact_cases.head()

,CaseID,OfficerID,DateCreated,Status,ResolutionHours,SLA_Target_Hours,SLA_Met_Flag,CSAT,CaseAge
0,1001,O004,2024-04-26,Closed,36,48,1.0,3.9,1.0
1,1002,O005,2024-04-24,Open,73,48,NaN,0.0,811.0
2,1003,O005,2024-04-12,Closed,60,48,0.0,4.7,2.0
3,1004,O003,2024-04-29,Closed,74,48,0.0,4.3,3.0
4,1005,O003,2024-04-03,Closed,16,48,1.0,4.3,0.0


In [8]:
# Create officer dimension table
dim_officer = df[
[
"OfficerID",
"OfficerName",
"Team"
]
].drop_duplicates()

In [9]:
# Check officer dimension table
dim_officer

,OfficerID,OfficerName,Team
0,O004,Officer 4,Team B
1,O005,Officer 5,Team C
3,O003,Officer 3,Team B
5,O001,Officer 1,Team A
6,O002,Officer 2,Team A


In [10]:
# Create status dimension table
dim_status = (
    df[["Status"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_status["StatusKey"] = dim_status.index + 1 # Add new column
dim_status = dim_status[["StatusKey", "Status"]] # Reorder columns

In [11]:
# Check status dimension table
dim_status

,StatusKey,Status
0,1,Closed
1,2,Open


In [12]:
# Merge status key into fact table
fact_cases = fact_cases.merge(
    dim_status,
    on="Status",
    how="left"
)

fact_cases.drop(columns="Status", inplace=True) # Drop status

In [13]:
# Check fact_cases table
fact_cases

,CaseID,OfficerID,DateCreated,ResolutionHours,SLA_Target_Hours,SLA_Met_Flag,CSAT,CaseAge,StatusKey
0,1001,O004,2024-04-26,36,48,1.0,3.9,1.0,1
1,1002,O005,2024-04-24,73,48,NaN,0.0,811.0,2
2,1003,O005,2024-04-12,60,48,0.0,4.7,2.0,1
3,1004,O003,2024-04-29,74,48,0.0,4.3,3.0,1
4,1005,O003,2024-04-03,16,48,1.0,4.3,0.0,1
...,...,...,...,...,...,...,...,...,...
195,1196,O004,2024-04-20,34,48,NaN,0.0,815.0,2
196,1197,O001,2024-04-22,71,48,0.0,4.6,2.0,1
197,1198,O003,2024-04-25,10,48,1.0,3.5,0.0,1
198,1199,O002,2024-04-15,72,48,0.0,4.1,3.0,1


In [14]:
# Create date dimension
dim_date = df[
[
"DateCreated",
"Year",
"Month",
"Week",
"Day"
]
].drop_duplicates()

dim_date = dim_date.sort_values("DateCreated") # Sort date

In [15]:
# Check dim_date
dim_date

,DateCreated,Year,Month,Week,Day
107,2024-04-01,2024,April,14,Monday
6,2024-04-02,2024,April,14,Tuesday
4,2024-04-03,2024,April,14,Wednesday
14,2024-04-04,2024,April,14,Thursday
23,2024-04-05,2024,April,14,Friday
16,2024-04-06,2024,April,14,Saturday
96,2024-04-07,2024,April,14,Sunday
67,2024-04-08,2024,April,15,Monday
9,2024-04-09,2024,April,15,Tuesday
11,2024-04-10,2024,April,15,Wednesday


In [16]:
# Check relationships: officer ids
fact_cases["OfficerID"].isin(dim_officer["OfficerID"]).all()

np.True_

In [17]:
# Check relationships: status keys
fact_cases["StatusKey"].isin(dim_status["StatusKey"]).all()

np.True_

In [18]:
# Check relationships: dates
fact_cases["DateCreated"].isin(dim_date["DateCreated"]).all()

np.True_

In [19]:
# Check primary keys: officer
dim_officer["OfficerID"].is_unique

True

In [20]:
# Check primary keys: status
dim_status["StatusKey"].is_unique

True

In [21]:
# Check primary keys: date
dim_date["DateCreated"].is_unique

True

In [22]:
# Check primary keys: case
fact_cases["CaseID"].is_unique

True

In [23]:
# Check misisng values
fact_cases.isnull().sum()

,0
CaseID,0
OfficerID,0
DateCreated,0
ResolutionHours,0
SLA_Target_Hours,0
SLA_Met_Flag,17
CSAT,0
CaseAge,0
StatusKey,0


In [24]:
# Check misisng values
dim_officer.isnull().sum()

,0
OfficerID,0
OfficerName,0
Team,0


In [25]:
# Check misisng values
dim_status.isnull().sum()

,0
StatusKey,0
Status,0


In [26]:
# Check misisng values
dim_date.isnull().sum()

,0
DateCreated,0
Year,0
Month,0
Week,0
Day,0


In [27]:
# Save fact table
fact_cases.to_csv("FactCases.csv", index=False)
files.download("FactCases.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
# Save officer table
dim_officer.to_csv("DimOfficer.csv", index=False)
files.download("DimOfficer.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
# Save status table
dim_status.to_csv("DimStatus.csv", index=False)
files.download("DimStatus.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# Check date table
dim_date.to_csv("DimDate.csv", index=False)
files.download("DimDate.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>